## Exploratory notebook setup

See `docs/notebooks.md` for inputs, execution order, and environment setup.
Run a working copy prepared by `tools/prepare_workspace.py`.
Saved outputs were removed from this copy; executed originals remain on the NAS.
The following cell changes file locations only. The analysis formulas are preserved.


In [ ]:
import os
from pathlib import Path

# Set these before launching Jupyter when using another storage mount.
VQ_NAS_ROOT = Path(os.environ.get("VQ_NAS_ROOT", "/megaverse/storage/nasiri"))
VQ_JUPYTER_ROOT = Path(os.environ.get("VQ_JUPYTER_ROOT", str(VQ_NAS_ROOT / "jupyter")))


In [ ]:
!pip install matplotlib pandas

# Global Noise robustness of image tokenizers: VQGAN vs LlamaGen

This section evaluates how sensitive some image tokenization pipelines are to additive Gaussian noise in image space. For each sample, we start from an original RGB image **x**, inject noise at several levels to get **x′**, run the codec (**encode → decode**) to obtain a reconstruction **x̂**, and then measure image-space distortion (PSNR) and token-space drift (how indices and global token usage change).

We aim to answer these questions:

- **How much of the final degradation is due to the noise itself vs the codec?** (via PSNR baselines, not just end-to-end PSNR)
- **How stable are tokens under input noise?** (flip rates per image, and changes in global token usage)
- **Does noise “kill” tokens or revive previously unused ones?** (dead/alive tracking and a clean vs noisy 2×2 summary)

## Data layout
Each sample directory is expected to contain these PNGs:
- `0_original.png`
- clean reconstruction: `1_recon_clean.png`
- noisy inputs and reconstructions:
  - low:  `2_input_noise_low.png`,  `3_recon_noise_low.png`
  - mid:  `4_input_noise_mid.png`,  `5_recon_noise_mid.png`
  - high: `6_input_noise_high.png`,  `7_recon_noise_high.png`

Token metadata is stored as sharded JSONL files:
- `metadata_part_*.jsonl` (one JSON object per sample, including token indices and noise level info)

## Outputs
- PSNR curves and summary tables across noise levels and models
- token flip-rate stats (per-image)
- global token usage comparisons (counts, entropy, dead tokens)
- final “dead vs alive” summary comparing clean vs noisy token usage


## Setup

In [ ]:
from __future__ import annotations

# --- stdlib ---
import json
import math
import os
import random
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, Iterable, Iterator, List, Optional, Tuple

# --- third-party ---
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

try:
    from tqdm.contrib.concurrent import process_map
except Exception:
    process_map = None


In [ ]:
SEED = 0

def set_seeds(seed: int = 0) -> None:
    random.seed(seed)
    np.random.seed(seed)

set_seeds(SEED)


In [ ]:
@dataclass(frozen=True)
class ExperimentConfig:
    # Noise levels used for x' = clip(x + sigma * eps)
    sigma_low: float = 0.1
    sigma_mid: float = 0.2
    sigma_high: float = 0.5

    # Codebook size (keep explicit; don't infer from observed max token id)
    codebook_size: int = 16384

    # Parallelism knobs (when scanning many samples / JSONL shards)
    max_workers: int = min(16, os.cpu_count() or 8)
    chunksize: int = 6

CFG = ExperimentConfig()

SIGMAS = np.array([0.0, CFG.sigma_low, CFG.sigma_mid, CFG.sigma_high], dtype=np.float64)
SIGMA_LABELS = ["clean (0.0)", f"low ({CFG.sigma_low})", f"mid ({CFG.sigma_mid})", f"high ({CFG.sigma_high})"]


In [ ]:
VQGAN_DIR    = VQ_JUPYTER_ROOT / "1765562245_robustness_dataset_vqgan"
LLAMAGEN_DIR = VQ_JUPYTER_ROOT / "1765559700_robustness_dataset_llamagen"

VQGAN_IMAGES    = VQGAN_DIR / "images"
LLAMAGEN_IMAGES = LLAMAGEN_DIR / "images"

print("VQGAN_DIR   :", VQGAN_DIR)
print("LLAMAGEN_DIR:", LLAMAGEN_DIR)
print("VQGAN images exists   :", VQGAN_IMAGES.exists())
print("LlamaGen images exists:", LLAMAGEN_IMAGES.exists())

In [ ]:
def find_sample_dirs(images_root: Path) -> List[Path]:
    if not images_root.exists():
        raise FileNotFoundError(f"Missing images directory: {images_root}")
    # Robust: walk all subdirs and find parents of 0_original.png
    sample_dirs = sorted({p.parent for p in images_root.rglob("0_original.png")})
    if not sample_dirs:
        raise RuntimeError(f"No samples found under {images_root} (expected files like 0_original.png).")
    return sample_dirs

vq_samples = find_sample_dirs(VQGAN_IMAGES)
ll_samples = find_sample_dirs(LLAMAGEN_IMAGES)

print("VQGAN samples   :", len(vq_samples))
print("LlamaGen samples:", len(ll_samples))
print("Example VQGAN sample dir   :", vq_samples[0])
print("Example LlamaGen sample dir:", ll_samples[0])

In [ ]:
METADATA_GLOB = "metadata_part_*.jsonl"

def find_metadata_files(run_dir: Path) -> List[Path]:
    files = sorted(run_dir.glob(METADATA_GLOB))
    if not files:
        raise FileNotFoundError(f"No metadata shards found under {run_dir} (glob {METADATA_GLOB}).")
    return files

def peek_first_jsonl(path: Path) -> Dict:
    with path.open("r") as f:
        for line in f:
            line = line.strip()
            if line:
                return json.loads(line)
    raise RuntimeError(f"Empty jsonl file: {path}")

vq_meta = find_metadata_files(VQGAN_DIR)
ll_meta = find_metadata_files(LLAMAGEN_DIR)

print("VQGAN metadata shards   :", len(vq_meta), "example:", vq_meta[0].name)
print("LlamaGen metadata shards:", len(ll_meta), "example:", ll_meta[0].name)

vq_first = peek_first_jsonl(vq_meta[0])
ll_first = peek_first_jsonl(ll_meta[0])

print("VQGAN first record keys   :", sorted(vq_first.keys()))
print("LlamaGen first record keys:", sorted(ll_first.keys()))


## Qualitative sanity check

In [ ]:
def load_img_float01(path: Path) -> np.ndarray:
    """Return RGB float32 image in [0,1] with shape (H,W,3)."""
    with Image.open(path) as im:
        im = im.convert("RGB")
        arr = np.asarray(im).astype(np.float32) / 255.0
    return arr

def delta_map(a: np.ndarray, b: np.ndarray) -> np.ndarray:
    """Per-pixel L2 difference map for visualization."""
    d = a - b
    return np.sqrt(np.sum(d * d, axis=-1))


In [ ]:
N_VIS = 6  # tweak freely
_rng = np.random.default_rng(SEED)

def pick_sample_dirs(sample_dirs: List[Path], n: int) -> List[Path]:
    n = min(n, len(sample_dirs))
    idx = _rng.choice(len(sample_dirs), size=n, replace=False)
    return [sample_dirs[i] for i in idx]

vq_vis = pick_sample_dirs(vq_samples, N_VIS)
ll_vis = pick_sample_dirs(ll_samples, N_VIS)

print("Picked VQGAN samples:")
for p in vq_vis[:3]:
    print(" ", p.name)
print("Picked LlamaGen samples:")
for p in ll_vis[:3]:
    print(" ", p.name)

In [ ]:
PANEL_INPUTS = [
    ("0_original.png",        "x (original)"),
    ("2_input_noise_low.png", "x′ low"),
    ("4_input_noise_mid.png", "x′ mid"),
    ("6_input_noise_high.png","x′ high"),
]

PANEL_RECONS = [
    ("1_recon_clean.png",     "x̃ = D(E(x))"),
    ("3_recon_noise_low.png", "x̂ low"),
    ("5_recon_noise_mid.png", "x̂ mid"),
    ("7_recon_noise_high.png","x̂ high"),
]

def show_grid(sample_dirs: List[Path], title: str, max_samples: int = 3) -> None:
    n = min(max_samples, len(sample_dirs))
    cols = 4

    # 3 rows per sample: label + inputs + recons
    rows = 3 * n

    fig, axes = plt.subplots(rows, cols, figsize=(2.2 * cols, 2.2 * rows))

    for i, sd in enumerate(sample_dirs[:n]):
        r_label = 3 * i
        r_inputs = r_label + 1
        r_recons = r_label + 2

        # --- label row (span visually across columns)
        for c in range(cols):
            axes[r_label, c].axis("off")
        axes[r_label, 0].text(
            0.5, 0.5, sd.name,
            transform=axes[r_label, 0].transAxes,
            ha="center", va="center", fontsize=11, fontweight="bold"
        )

        # --- inputs
        for c, (fn, lab) in enumerate(PANEL_INPUTS):
            ax = axes[r_inputs, c]
            ax.imshow(load_img_float01(sd / fn))
            ax.axis("off")
            ax.set_title(lab, fontsize=10)

        # --- reconstructions
        for c, (fn, lab) in enumerate(PANEL_RECONS):
            ax = axes[r_recons, c]
            ax.imshow(load_img_float01(sd / fn))
            ax.axis("off")
            ax.set_title(lab, fontsize=10)

    fig.suptitle(title, fontsize=14)
    plt.tight_layout()
    plt.show()


show_grid(vq_vis, "VQGAN: qualitative sanity check")
show_grid(ll_vis, "LlamaGen: qualitative sanity check")

## Metrics and formalization

We study robustness to additive Gaussian noise in image space.

Let $x$ be the original RGB image (normalized to $[0,1]$). We inject noise as:
$$
x' = \mathrm{clip}(x + \sigma \cdot \varepsilon), \quad \varepsilon \sim \mathcal{N}(0, I)
$$
and pass the noisy image through the codec:
$$
\hat{x} = D(E(x'))
$$
where $E$ is the encoder (image $\to$ tokens) and $D$ is the decoder (tokens $\to$ image).

### Why we need baselines
The end-to-end metric $PSNR(x, \hat{x})$ mixes two effects:
1) distortion from injected noise ($\sigma$ and clipping)
2) distortion from the codec ($E$ and $D$)

So we compute a small set of complementary PSNR baselines:

**(A) Noise-only distortion**
$$
PSNR(x, x')
$$

**(B) Codec distortion at the noisy point**
$$
PSNR(x', \hat{x})
$$

**(C) End-to-end distortion vs the clean original**
$$
PSNR(x, \hat{x})
$$

**(D) Clean reconstruction baseline (codec-only at $\sigma=0$)**
$$
\tilde{x} = D(E(x)), \quad PSNR(x, \tilde{x})
$$

All PSNRs are computed in RGB on $[0,1]$ with $MAX_I = 1$.


In [ ]:
def mse(a: np.ndarray, b: np.ndarray) -> float:
    a = a.astype(np.float32)
    b = b.astype(np.float32)
    return float(np.mean((a - b) ** 2))

def psnr(a: np.ndarray, b: np.ndarray, max_i: float = 1.0) -> float:
    m = mse(a, b)
    if m == 0.0:
        return float("inf")
    return 10.0 * math.log10((max_i ** 2) / m)

In [ ]:
LEVEL_TO_FILES = {
    "clean": ("0_original.png", "1_recon_clean.png", None),   # (x, x̃, x')
    "low":   ("0_original.png", "3_recon_noise_low.png", "2_input_noise_low.png"),
    "mid":   ("0_original.png", "5_recon_noise_mid.png", "4_input_noise_mid.png"),
    "high":  ("0_original.png", "7_recon_noise_high.png", "6_input_noise_high.png"),
}

def load_triplet(sample_dir: Path, level: str) -> Tuple[np.ndarray, np.ndarray, Optional[np.ndarray]]:
    """
    Returns (x, x_recon, x_noisy_input or None for clean level).
    """
    x_fn, recon_fn, noisy_fn = LEVEL_TO_FILES[level]
    x = load_img_float01(sample_dir / x_fn)
    x_recon = load_img_float01(sample_dir / recon_fn)
    x_noisy = load_img_float01(sample_dir / noisy_fn) if noisy_fn is not None else None
    return x, x_recon, x_noisy

In [ ]:
LEVEL_ORDER = ["clean", "low", "mid", "high"]
LEVEL_TO_SIGMA = {
    "clean": 0.0,
    "low": CFG.sigma_low,
    "mid": CFG.sigma_mid,
    "high": CFG.sigma_high,
}

def compute_psnr_row(sample_dir: Path, model: str, level: str) -> Dict:
    x, x_recon, x_noisy = load_triplet(sample_dir, level)

    row = {
        "model": model,
        "sample_id": sample_dir.name,
        "level": level,
        "sigma": LEVEL_TO_SIGMA[level],
        # PSNR baselines (some may be NaN depending on availability)
        "psnr_x_xnoisy": np.nan,
        "psnr_xnoisy_xhat": np.nan,
        "psnr_x_xhat": np.nan,
        "psnr_x_xcleanrecon": np.nan,
    }

    if level == "clean":
        # D: PSNR(x, x̃)
        row["psnr_x_xcleanrecon"] = psnr(x, x_recon)
        # For completeness, also set xhat meaning here
        row["psnr_x_xhat"] = row["psnr_x_xcleanrecon"]
    else:
        # A: PSNR(x, x')
        row["psnr_x_xnoisy"] = psnr(x, x_noisy)
        # B: PSNR(x', x̂)
        row["psnr_xnoisy_xhat"] = psnr(x_noisy, x_recon)
        # C: PSNR(x, x̂)
        row["psnr_x_xhat"] = psnr(x, x_recon)

    return row

In [ ]:
# smoke test on one sample (sanity check numbers exist)

_test_sd = vq_samples[0]
rows = [compute_psnr_row(_test_sd, "vqgan", lvl) for lvl in LEVEL_ORDER]
pd.DataFrame(rows)

### Compute PSNR baselines over the full dataset

In [ ]:
CACHE_DIR = Path("./analysis_cache")  # local to the notebook working dir
CACHE_DIR.mkdir(parents=True, exist_ok=True)

def psnr_cache_path(model: str) -> Path:
    # Include key config bits in the filename so caches don't silently mismatch.
    tag = f"sL{CFG.sigma_low}_sM{CFG.sigma_mid}_sH{CFG.sigma_high}_K{CFG.codebook_size}"
    return CACHE_DIR / f"psnr_baselines__{model}__{tag}.parquet"

In [ ]:
# Save/load helpers (parquet if available, otherwise CSV fallback)

def save_df(df: pd.DataFrame, path: Path) -> None:
    path = Path(path)
    try:
        df.to_parquet(path, index=False)
    except Exception:
        csv_path = path.with_suffix(".csv")
        df.to_csv(csv_path, index=False)
        print(f"[save_df] Parquet failed; saved CSV instead: {csv_path}")
        return
    print(f"[save_df] Saved: {path}")

def load_df(path: Path) -> pd.DataFrame:
    path = Path(path)
    if path.exists():
        return pd.read_parquet(path)
    csv_path = path.with_suffix(".csv")
    if csv_path.exists():
        return pd.read_csv(csv_path)
    raise FileNotFoundError(f"No cache found at {path} or {csv_path}")


In [ ]:
# Fast full-dataset computation (parallel per-sample, notebook-safe)

MAX_WORKERS = CFG.max_workers
CHUNKSIZE   = 64

def psnr_rows_for_sample_dir(args) -> List[Dict]:
    """Worker: compute all levels for one sample dir with minimal IO."""
    sample_dir, model = args

    # load shared original once
    x = load_img_float01(sample_dir / "0_original.png")

    # clean recon
    x_cleanrecon = load_img_float01(sample_dir / "1_recon_clean.png")
    rows = [{
        "model": model,
        "sample_id": sample_dir.name,
        "level": "clean",
        "sigma": 0.0,
        "psnr_x_xnoisy": np.nan,
        "psnr_xnoisy_xhat": np.nan,
        "psnr_x_xhat": psnr(x, x_cleanrecon),
        "psnr_x_xcleanrecon": psnr(x, x_cleanrecon),
    }]

    # noisy levels in one pass
    for level, sigma, noisy_fn, recon_fn in [
        ("low",  CFG.sigma_low,  "2_input_noise_low.png",  "3_recon_noise_low.png"),
        ("mid",  CFG.sigma_mid,  "4_input_noise_mid.png",  "5_recon_noise_mid.png"),
        ("high", CFG.sigma_high, "6_input_noise_high.png", "7_recon_noise_high.png"),
    ]:
        x_noisy = load_img_float01(sample_dir / noisy_fn)
        x_hat   = load_img_float01(sample_dir / recon_fn)

        rows.append({
            "model": model,
            "sample_id": sample_dir.name,
            "level": level,
            "sigma": float(sigma),
            "psnr_x_xnoisy": psnr(x, x_noisy),         # A
            "psnr_xnoisy_xhat": psnr(x_noisy, x_hat),  # B
            "psnr_x_xhat": psnr(x, x_hat),             # C
            "psnr_x_xcleanrecon": np.nan,
        })

    return rows


def compute_psnr_df(sample_dirs: List[Path], model: str) -> pd.DataFrame:
    tasks = [(sd, model) for sd in sample_dirs]

    if process_map is None:
        # fallback sequential
        rows_nested = [psnr_rows_for_sample_dir(t) for t in tasks]
    else:
        rows_nested = process_map(
            psnr_rows_for_sample_dir,
            tasks,
            max_workers=MAX_WORKERS,
            chunksize=CHUNKSIZE,
            desc=f"PSNR {model}",
        )

    # flatten list-of-lists
    rows = [r for sub in rows_nested for r in sub]
    df = pd.DataFrame(rows).sort_values(["model", "sample_id", "sigma"]).reset_index(drop=True)
    return df


In [ ]:
# Compute-or-load caches

def get_or_compute_psnr(model: str, sample_dirs: List[Path], force: bool = False) -> pd.DataFrame:
    path = psnr_cache_path(model)
    if (not force) and (path.exists() or path.with_suffix(".csv").exists()):
        print(f"Loading cached PSNR table for {model}...")
        return load_df(path)

    print(f"Computing PSNR table for {model}...")
    df = compute_psnr_df(sample_dirs, model)
    save_df(df, path)
    return df

psnr_vq = get_or_compute_psnr("vqgan", vq_samples, force=False)
psnr_ll = get_or_compute_psnr("llamagen", ll_samples, force=False)

print(psnr_vq.shape, psnr_ll.shape)
psnr_vq.head(3)

## Image-space results: summaries and comparable plots

In [ ]:
psnr_vq = load_df(psnr_cache_path("vqgan"))
psnr_ll = load_df(psnr_cache_path("llamagen"))
psnr_all = pd.concat([psnr_vq, psnr_ll], ignore_index=True)

psnr_all.shape


In [ ]:
METRICS = ["psnr_x_xnoisy", "psnr_xnoisy_xhat", "psnr_x_xhat", "psnr_x_xcleanrecon"]

def summarize_psnr(df: pd.DataFrame) -> pd.DataFrame:
    g = df.groupby(["model", "level", "sigma"], sort=True)
    out = g[METRICS].agg(["mean", "std", "median", "min", "max"]).reset_index()

    # flatten metric columns only (leave model/level/sigma untouched)
    out.columns = [
        col if isinstance(col, str) else (col[0] if col[1] == "" else f"{col[0]}_{col[1]}")
        for col in out.columns
    ]

    return out.sort_values(["model", "sigma", "level"]).reset_index(drop=True)


psnr_summary = summarize_psnr(psnr_all)
psnr_summary.head(12)

**Summary table (mean/std/median/min/max by model and sigma)**

This table aggregates the three PSNR baselines plus the clean reconstruction reference. It provides the numeric counterpart to the plots: central tendency (mean/median), variability (std), and extremes (min/max) for each $(\text{model}, \sigma)$ group. In particular, comparing $PSNR(x,x')$ to $PSNR(x',\hat{x})$ and $PSNR(x,\hat{x})$ lets us separate “how strong the corruption is” from “how much extra damage the codec adds” at each noise level.


In [ ]:
# One-row plot: per model, show 3 curves + clean baseline (with std)

METRIC_INFO = {
    "psnr_x_xnoisy":    ("PSNR(x, x')",        "Noise-only"),
    "psnr_xnoisy_xhat": ("PSNR(x', x̂)",       "Codec @ noisy"),
    "psnr_x_xhat":      ("PSNR(x, x̂)",        "End-to-end"),
}

BASELINE_METRIC = "psnr_x_xcleanrecon"  # only defined for level==clean

def _mean_std_by_model_sigma(df: pd.DataFrame, metric: str) -> pd.DataFrame:
    d = df[["model", "sigma", metric]].copy()
    g = d.groupby(["model", "sigma"], sort=True)[metric]
    out = g.agg(["mean", "std"]).reset_index()
    return out

def _baseline_by_model(df: pd.DataFrame) -> Dict[str, float]:
    # baseline is only defined at clean rows
    d = df[df["level"] == "clean"].groupby("model")[BASELINE_METRIC].mean()
    return d.to_dict()

def plot_three_curves_plus_baseline(df: pd.DataFrame, title: str) -> None:
    models = ["vqgan", "llamagen"]
    sigmas = sorted(df["sigma"].unique())
    baseline = _baseline_by_model(df)

    # precompute summaries for y-lims (include +- std)
    summaries = {k: _mean_std_by_model_sigma(df, k) for k in METRIC_INFO.keys()}

    all_y = []
    for k, sm in summaries.items():
        sm2 = sm.replace([np.inf, -np.inf], np.nan).dropna()
        all_y.append((sm2["mean"] - sm2["std"]).values)
        all_y.append((sm2["mean"] + sm2["std"]).values)
    # add baselines too
    all_y.append(np.array(list(baseline.values()), dtype=float))

    all_y = np.concatenate(all_y) if len(all_y) else np.array([0.0])
    ylo, yhi = float(np.min(all_y)), float(np.max(all_y))
    pad = 0.05 * (yhi - ylo + 1e-9)
    ylims = (ylo - pad, yhi + pad)

    fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=True)

    for ax, model in zip(axes, models):
        for metric, (pretty, short) in METRIC_INFO.items():
            sm = summaries[metric]
            mm = sm[sm["model"] == model].sort_values("sigma")

            ax.plot(mm["sigma"], mm["mean"], marker="o", label=short)
            ax.fill_between(
                mm["sigma"],
                mm["mean"] - mm["std"],
                mm["mean"] + mm["std"],
                alpha=0.12,
            )

        # clean baseline as horizontal line
        if model in baseline:
            ax.axhline(baseline[model], linestyle="--", linewidth=1.5, label="Clean baseline PSNR(x,x̃)")

        ax.set_title(model)
        ax.set_xlabel("sigma")
        ax.set_ylabel("PSNR (dB)")
        ax.set_ylim(*ylims)
        ax.set_xticks(sigmas, [f"{s:g}" for s in sigmas])
        ax.grid(True, alpha=0.3)
        ax.legend()

    fig.suptitle(title)
    plt.tight_layout()
    plt.show()

plot_three_curves_plus_baseline(psnr_all, "PSNR baselines vs input noise (mean ± std)")


**PSNR baselines vs input noise (mean ± std)**

Each panel decomposes end-to-end degradation into two sources: the distortion introduced by the noise itself (blue, $PSNR(x,x')$) and the additional loss introduced by the codec when encoding/decoding a noisy input (orange, $PSNR(x',\hat{x})$). The green curve is the end-to-end quality relative to the clean original ($PSNR(x,\hat{x})$). The dashed horizontal line is the clean reconstruction baseline ($PSNR(x,\tilde{x})$ at $\sigma=0$), shown as a reference level for “codec-only” distortion. As expected, all PSNRs drop monotonically with increasing $\sigma$, and the gap between the green curve and the dashed baseline shows how quickly robustness deteriorates relative to clean reconstruction quality.


In [ ]:
# Overlay plot: one metric, both models, mean ± std

def plot_metric_overlay_models(df: pd.DataFrame, metric: str, title: str, show_clean_baseline: bool = False) -> None:
    sm = _mean_std_by_model_sigma(df, metric).replace([np.inf, -np.inf], np.nan).dropna()

    sigmas = sorted(sm["sigma"].unique())
    models = ["vqgan", "llamagen"]
    baseline = _baseline_by_model(df) if show_clean_baseline else {}

    # global y-lims (include std)
    ylo = float(np.min((sm["mean"] - sm["std"]).values))
    yhi = float(np.max((sm["mean"] + sm["std"]).values))
    if baseline:
        ylo = min(ylo, min(baseline.values()))
        yhi = max(yhi, max(baseline.values()))
    pad = 0.05 * (yhi - ylo + 1e-9)
    ylims = (ylo - pad, yhi + pad)

    plt.figure(figsize=(10, 5))
    for model in models:
        mm = sm[sm["model"] == model].sort_values("sigma")
        plt.errorbar(
            mm["sigma"], mm["mean"], yerr=mm["std"],
            fmt='-o', linewidth=2, capsize=5, elinewidth=1.2,
            label=model
        )
        plt.fill_between(mm["sigma"], mm["mean"] - mm["std"], mm["mean"] + mm["std"], alpha=0.10)

        if show_clean_baseline and model in baseline:
            plt.axhline(baseline[model], linestyle="--", linewidth=1.2, alpha=0.8)

    plt.title(title)
    plt.xlabel("Gaussian noise std in normalized space (sigma)")
    plt.ylabel("PSNR (dB)")
    plt.xticks(sigmas, [f"{s:g}" for s in sigmas])
    plt.ylim(*ylims)
    plt.grid(True, alpha=0.35)
    plt.legend()
    plt.tight_layout()
    plt.show()

# Examples
plot_metric_overlay_models(psnr_all, "psnr_x_xhat",      "End-to-end PSNR(x, x̂) vs sigma (mean ± std)", show_clean_baseline=True)
plot_metric_overlay_models(psnr_all, "psnr_x_xnoisy",    "Noise-only PSNR(x, x') vs sigma (mean ± std)")
plot_metric_overlay_models(psnr_all, "psnr_xnoisy_xhat", "Codec PSNR(x', x̂) vs sigma (mean ± std)")


**End-to-end PSNR(x, x̂) vs sigma (mean ± std)**

This plot compares the overall robustness of the two pipelines under input noise by measuring $PSNR(x,\hat{x})$ as $\sigma$ increases. Error bars (±1 std) highlight that performance varies substantially across images even at fixed $\sigma$, but the trend is consistent: higher noise pushes reconstructions further from the clean original. The dashed horizontal references indicate the clean reconstruction baselines ($\sigma=0$) for each model, which helps interpret how much of the drop comes from the noise sensitivity rather than intrinsic codec distortion.

**Noise-only PSNR(x, x′) vs sigma (mean ± std)**

This is the calibration baseline: it measures how much the injected noise (plus clipping) alters the image before any encoding/decoding. Both runs show essentially the same curve because the noise process is applied in image space and does not depend on the tokenizer. This baseline makes the other PSNR curves interpretable: e.g., a given $\sigma$ corresponds to a specific “raw” corruption level even before the codec is applied.

**Codec PSNR(x′, x̂) vs sigma (mean ± std)**

This plot isolates codec behavior at each noise level by comparing the noisy input $x'$ to its reconstruction $\hat{x}=D(E(x'))$. As $\sigma$ increases, the codec introduces progressively more distortion relative to the already-corrupted input, indicating that the encode→decode pipeline is not invariant to small pixel-space perturbations. Comparing models here is useful because it removes the “noise-only” component and focuses on how well the codec preserves a noisy observation.

In [ ]:
# distribution view at each sigma

def plot_metric_hist(df: pd.DataFrame, metric: str, bins: int = 60) -> None:
    d = df[["model", "sigma", metric]].copy()
    d = d.replace([np.inf, -np.inf], np.nan).dropna()

    sigmas = sorted(d["sigma"].unique())
    models = ["vqgan", "llamagen"]

    # global x-lims for comparability across subplots
    xvals = d[metric].values
    xlo, xhi = float(np.min(xvals)), float(np.max(xvals))
    pad = 0.03 * (xhi - xlo + 1e-9)
    xlims = (xlo - pad, xhi + pad)

    fig, axes = plt.subplots(len(models), len(sigmas), figsize=(3.5 * len(sigmas), 3.0 * len(models)), sharex=True, sharey=True)

    for i, model in enumerate(models):
        for j, s in enumerate(sigmas):
            ax = axes[i, j] if len(models) > 1 else axes[j]
            dd = d[(d["model"] == model) & (d["sigma"] == s)][metric]
            ax.hist(dd.values, bins=bins)
            ax.set_title(f"{model}, σ={s:g}")
            ax.set_xlim(*xlims)
            ax.grid(True, alpha=0.2)

    fig.suptitle(f"Distribution of {metric} across sigma (shared axes)")
    plt.tight_layout()
    plt.show()

# Example: end-to-end PSNR distribution
plot_metric_hist(psnr_all, "psnr_x_xhat")


**Distribution of end-to-end PSNR across sigma (shared axes)**

Histograms show the full distribution of $PSNR(x,\hat{x})$ over images for each $\sigma$, using shared axes to make shifts directly comparable. As noise increases, the distribution shifts left (lower PSNR) and typically tightens at high noise, meaning most samples collapse to a similar low-quality regime. This view is more informative than means alone: it reveals tails and heterogeneity, i.e., whether degradation is uniform across the dataset or driven by a subset of particularly fragile images.


## Token-space robustness

Each sample provides token grids:
- clean tokens: $z$
- noisy tokens: $z'(\sigma)$ for $\sigma \in \{0.1, 0.2, 0.5\}$

We measure:
1) **Token flip rate** per image: $\text{flip}(\sigma) = \frac{1}{HW}\sum_{i,j}\mathbb{1}[z_{ij}\neq z'_{ij}(\sigma)]$
2) **Global token usage**: counts per token-id across the dataset, per noise level
3) **Dead-token confusion** (per token-id): whether a token has zero count at clean vs noisy


In [ ]:
from pathlib import Path

CACHE_DIR = Path("analysis_cache")
CACHE_DIR.mkdir(exist_ok=True)

LEVELS = [
    ("clean", 0.0, "indices_clean"),
    ("low",   0.1, "indices_low"),
    ("mid",   0.2, "indices_mid"),
    ("high",  0.5, "indices_high"),
]

In [ ]:
import json
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

def token_flip_table(meta_parts, model_name: str, cache_path: Path) -> pd.DataFrame:
    if cache_path.exists():
        return pd.read_csv(cache_path)

    rows = []
    for p in tqdm(meta_parts, desc=f"token flip {model_name}"):
        with p.open("r") as f:
            for line in f:
                if not line.strip():
                    continue
                obj = json.loads(line)
                z0 = np.asarray(obj["indices_clean"], dtype=np.int32)

                for level, sigma, key in LEVELS:
                    if level == "clean":
                        continue
                    z = np.asarray(obj[key], dtype=np.int32)
                    flip = float((z != z0).mean())
                    rows.append({
                        "model": model_name,
                        "sample_id": obj["image_id"],
                        "level": level,
                        "sigma": sigma,
                        "token_flip_frac": flip,
                    })

    df = pd.DataFrame(rows)
    df.to_csv(cache_path, index=False)
    return df

flip_vq = token_flip_table(vq_meta, "vqgan",    CACHE_DIR / "token_flip__vqgan.csv")
flip_ll = token_flip_table(ll_meta, "llamagen", CACHE_DIR / "token_flip__llamagen.csv")
flip_all = pd.concat([flip_vq, flip_ll], ignore_index=True)

flip_all.head()

**Token flip table (per-image)**

For each image and noise level, `token_flip_frac` reports the fraction of token positions that change compared to the clean token grid:
$$
\text{flip}(\sigma) = \frac{1}{HW}\sum_{i,j}\mathbb{1}[z_{ij}\neq z'_{ij}(\sigma)].
$$
Values close to 0 mean the discrete representation is stable under noise; values close to 1 mean the tokenization is almost completely different. In the shown examples, even $\sigma=0.1$ already flips a majority of tokens (≈0.60–0.70), and by $\sigma=0.5$ the flip rate reaches 1.0 (every token differs), indicating extreme sensitivity of the discrete token grid to pixel-space noise.


In [ ]:
def plot_flip_vs_sigma(flip_df: pd.DataFrame) -> None:
    g = (flip_df.groupby(["model", "sigma"])["token_flip_frac"]
         .agg(["mean","std"]).reset_index())

    plt.figure(figsize=(10, 5))
    for model in ["vqgan", "llamagen"]:
        m = g[g["model"] == model].sort_values("sigma")
        plt.errorbar(m["sigma"], m["mean"], yerr=m["std"], marker="o", capsize=5, label=model)

    plt.title("Token flip rate vs input noise (mean ± std)")
    plt.xlabel("sigma")
    plt.ylabel("fraction of tokens changed")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()

plot_flip_vs_sigma(flip_all)

**Token flip rate vs input noise (mean ± std)**

This plot quantifies how stable the discrete token grid is under pixel-space noise. Even mild noise ((\sigma=0.1)) already changes a large fraction of tokens (roughly three quarters for VQGAN and even more for LlamaGen on average), and by (\sigma=0.2) both are near “most tokens flipped”. At (\sigma=0.5) the flip rate is ~1.0, meaning the tokenization is essentially a different code for almost every position. The error bars show substantial image-to-image variability at low noise (some images are much more stable than others), but that variability collapses at high noise because nearly everything flips for everyone. The key takeaway: the tokenizer is highly sensitive to additive noise, so any apparent robustness in decoded images must come from the decoder smoothing/denoising, not from token stability.


In [ ]:
# Helper: global token counts per level (chunked bincount)

def token_counts_from_parts(meta_parts: List[Path], codebook_size: int, chunk_lines: int = 2000) -> Dict[str, np.ndarray]:
    level_map = [
        ("Clean (0.0)", "indices_clean"),
        (f"Low ({CFG.sigma_low})", "indices_low"),
        (f"Mid ({CFG.sigma_mid})", "indices_mid"),
        (f"High ({CFG.sigma_high})", "indices_high"),
    ]
    counts = {label: np.zeros(codebook_size, dtype=np.int64) for label, _ in level_map}

    for p in meta_parts:
        buffers = {label: [] for label, _ in level_map}

        with p.open("r") as f:
            for line in f:
                if not line.strip():
                    continue
                obj = json.loads(line)

                for label, key in level_map:
                    buffers[label].append(obj[key])

                # flush
                if len(buffers[level_map[0][0]]) >= chunk_lines:
                    for label, _ in level_map:
                        flat = np.array(buffers[label], dtype=np.int32).ravel()
                        counts[label] += np.bincount(flat, minlength=codebook_size)
                        buffers[label].clear()

        # flush remainder
        for label, _ in level_map:
            if buffers[label]:
                flat = np.array(buffers[label], dtype=np.int32).ravel()
                counts[label] += np.bincount(flat, minlength=codebook_size)
                buffers[label].clear()

    return counts


In [ ]:
import numpy as np

def counts_cache_path(model_name: str) -> Path:
    return CACHE_DIR / f"token_counts__{model_name}.npz"

def compute_or_load_counts(meta_parts, model_name: str):
    path = counts_cache_path(model_name)
    if path.exists():
        data = np.load(path, allow_pickle=True)
        return {k: data[k] for k in data.files}

    codebook = CFG.codebook_size
    counts = token_counts_from_parts(meta_parts, codebook_size=codebook, chunk_lines=2000)  #:contentReference[oaicite:5]{index=5}

    np.savez_compressed(path, **{label: vec for label, vec in counts.items()})
    return counts

vq_counts = compute_or_load_counts(vq_meta, "vqgan")
ll_counts = compute_or_load_counts(ll_meta, "llamagen")

In [ ]:
def dead_confusion(counts_clean: np.ndarray, counts_noisy: np.ndarray) -> pd.DataFrame:
    dead_c = (counts_clean == 0)
    dead_n = (counts_noisy == 0)

    # rows = clean (alive/dead), cols = noisy (alive/dead)
    aa = int((~dead_c & ~dead_n).sum())  # alive -> alive
    ad = int((~dead_c &  dead_n).sum())  # alive -> dead
    da = int(( dead_c & ~dead_n).sum())  # dead  -> alive
    dd = int(( dead_c &  dead_n).sum())  # dead  -> dead

    mat = pd.DataFrame(
        [[aa, ad],
         [da, dd]],
        index=["clean_alive", "clean_dead"],
        columns=["noisy_alive", "noisy_dead"]
    )
    return mat

def dead_confusion_for_model(model_name: str, counts_dict: dict) -> dict:
    out = {}
    clean = counts_dict["Clean (0.0)"]
    for label in ["Low (0.1)", "Mid (0.2)", "High (0.5)"]:
        out[label] = dead_confusion(clean, counts_dict[label])
    return out

dead_cm_ll = dead_confusion_for_model("llamagen", ll_counts)
dead_cm_vq = dead_confusion_for_model("vqgan", vq_counts)

In [ ]:
def show_dead_confusions(model_name: str, counts_dict: dict) -> None:
    print(f"\n{model_name} dead/non-dead confusion (token-id level)")
    clean = counts_dict["Clean (0.0)"]
    for label in [f"Low ({CFG.sigma_low})", f"Mid ({CFG.sigma_mid})", f"High ({CFG.sigma_high})"]:
        cm = dead_confusion(clean, counts_dict[label])
        print(f"\nClean vs {label}")
        display(cm)

show_dead_confusions("VQGAN", vq_counts)
show_dead_confusions("LlamaGen", ll_counts)


In [ ]:
# clean counts for VQGAN
vq_clean = vq_counts["Clean (0.0)"]

# alive token indices
alive_token_indices = np.nonzero(vq_clean)[0]
# equivalent:
# alive_token_indices = np.where(vq_clean > 0)[0]

In [ ]:
alive_token_indices

A token id is “alive” at a level if it appears at least once anywhere in the dataset at that level (global count > 0)



In [ ]:
# token usage visualization

TOPK = 100
COUNT_LABELS = [
    "Clean (0.0)",
    f"Low ({CFG.sigma_low})",
    f"Mid ({CFG.sigma_mid})",
    f"High ({CFG.sigma_high})",
]


def topk_tokens(counts_vec: np.ndarray, k: int) -> Tuple[np.ndarray, np.ndarray]:
    idx = np.argsort(counts_vec)[::-1][:k]
    return idx, counts_vec[idx]

def plot_topk_stacked(model_name: str, counts_dict: Dict[str, np.ndarray], topk: int = 150) -> None:
    fig, axes = plt.subplots(len(COUNT_LABELS), 1, figsize=(18, 3.6 * len(COUNT_LABELS)), sharex=True)
    if len(COUNT_LABELS) == 1:
        axes = [axes]

    # compute global y-limit across all panels (so you can compare levels)
    ymax = 0
    top_data = {}
    for label in COUNT_LABELS:
        idx, val = topk_tokens(counts_dict[label], k=topk)
        top_data[label] = (idx, val)
        ymax = max(ymax, int(val.max()) if len(val) else 0)

    for r, label in enumerate(COUNT_LABELS):
        idx, val = top_data[label]
        ax = axes[r]
        ax.bar(np.arange(topk), val)
        ax.set_title(f"{model_name} | Top-{topk} tokens | {label}")
        ax.set_ylabel("count")
        ax.set_ylim(0, ymax * 1.05)
        ax.grid(True, axis="y", linestyle="--", alpha=0.35)

        ax.set_xticks(np.arange(topk))
        ax.set_xticklabels([str(i) for i in idx], rotation=90, fontsize=7)

    axes[-1].set_xlabel("token id (sorted by frequency, top-k)")
    plt.tight_layout()
    plt.show()

plot_topk_stacked("VQGAN", vq_counts, topk=TOPK)
plot_topk_stacked("LlamaGen", ll_counts, topk=TOPK)


In [ ]:
# Full frequency curves (all tokens, rank-frequency), shared y-scale across levels

def plot_full_frequency_curve(model_name: str, counts_dict: Dict[str, np.ndarray], logy: bool = False) -> None:
    fig, axes = plt.subplots(len(COUNT_LABELS), 1, figsize=(12, 3.2 * len(COUNT_LABELS)), sharex=True, sharey=True)
    if len(COUNT_LABELS) == 1:
        axes = [axes]

    # global y-limit for comparability
    ymax = 0
    curves = {}
    for label in COUNT_LABELS:
        y = np.sort(counts_dict[label])[::-1]  # descending counts
        curves[label] = y
        ymax = max(ymax, int(y.max()) if len(y) else 0)

    for r, label in enumerate(COUNT_LABELS):
        y = curves[label]
        x = np.arange(1, len(y) + 1)

        ax = axes[r]
        ax.plot(x, y, linewidth=1.6)
        ax.set_title(f"{model_name} | Token rank-frequency | {label}")
        ax.set_ylabel("count")
        ax.set_ylim(0, ymax * 1.05)
        if logy:
            ax.set_yscale("log")
            ax.set_xscale("log")
        ax.grid(True, linestyle="--", alpha=0.35)

    axes[-1].set_xlabel("token rank (sorted by frequency, descending)")
    plt.tight_layout()
    plt.show()

plot_full_frequency_curve("VQGAN", vq_counts, logy=False)
plot_full_frequency_curve("LlamaGen", ll_counts, logy=False)

# optional: log-y reveals tail structure (often more informative)
plot_full_frequency_curve("VQGAN", vq_counts, logy=True)
plot_full_frequency_curve("LlamaGen", ll_counts, logy=True)


In [ ]:
vq_counts

In [ ]:
ll_counts

In [ ]:
LEVEL_KEYS = [
    ("Clean (0.0)", "indices_clean"),
    (f"Low ({CFG.sigma_low})",   "indices_low"),
    (f"Mid ({CFG.sigma_mid})",   "indices_mid"),
    (f"High ({CFG.sigma_high})", "indices_high"),
]

def infer_tokens_per_image(meta_parts: List[Path]) -> int:
    # look at the first non-empty jsonl line
    for p in meta_parts:
        with p.open("r") as f:
            for line in f:
                if line.strip():
                    obj = json.loads(line)
                    return int(len(obj["indices_clean"]))
    raise RuntimeError("Could not infer tokens_per_image (metadata empty).")

def count_images(meta_parts: List[Path]) -> int:
    n = 0
    for p in meta_parts:
        with p.open("r") as f:
            for line in f:
                if line.strip():
                    n += 1
    return n

def global_token_stats(meta_parts: List[Path], name: str) -> None:
    seen = {label: np.zeros(CFG.codebook_size, dtype=bool) for label, _ in LEVEL_KEYS}
    total = {label: 0 for label, _ in LEVEL_KEYS}
    min_id = {label: +10**9 for label, _ in LEVEL_KEYS}
    max_id = {label: -1 for label, _ in LEVEL_KEYS}

    n_images = count_images(meta_parts)
    tpi = infer_tokens_per_image(meta_parts)
    expected_total = n_images * tpi

    for p in meta_parts:
        with p.open("r") as f:
            for line in f:
                if not line.strip():
                    continue
                obj = json.loads(line)
                for label, key in LEVEL_KEYS:
                    arr = np.asarray(obj[key], dtype=np.int32)
                    total[label] += arr.size
                    mi = int(arr.min()); ma = int(arr.max())
                    if mi < min_id[label]: min_id[label] = mi
                    if ma > max_id[label]: max_id[label] = ma
                    seen[label][np.unique(arr)] = True

    print(f"\n{name} global token stats")
    print(f"Images: {n_images} | tokens/image: {tpi} | expected tokens/level: {expected_total}")
    print(f"{'Level':<16} | {'total':>10} | {'unique':>6} | {'dead':>6} | {'min':>5} | {'max':>5}")
    print("-" * 66)

    for label, _ in LEVEL_KEYS:
        uniq = int(seen[label].sum())
        dead = CFG.codebook_size - uniq
        print(f"{label:<16} | {total[label]:10d} | {uniq:6d} | {dead:6d} | {min_id[label]:5d} | {max_id[label]:5d}")

# Run for both
global_token_stats(vq_meta, "VQGAN")
global_token_stats(ll_meta, "LlamaGen")


In [ ]:
# Entropy + Perplexity + Top-k mass (from global count vectors)

TOPK_LIST = [10, 100, 1000]

def token_distribution_stats(counts_vec: np.ndarray, topk_list=TOPK_LIST) -> Dict[str, float]:
    counts = counts_vec.astype(np.float64)
    total = float(counts.sum())
    if total == 0:
        raise ValueError("counts_vec sums to zero; cannot form a distribution.")

    p = counts / total

    nonzero = int((counts > 0).sum())
    dead = int((counts == 0).sum())
    util_pct = 100.0 * (nonzero / counts.size)

    p_nz = p[p > 0]
    entropy = float(-(p_nz * np.log(p_nz)).sum())   # nats
    perplexity = float(np.exp(entropy))

    sorted_p = np.sort(p)[::-1]
    top_mass = {k: float(sorted_p[:k].sum()) for k in topk_list}

    out = {
        "total_tokens": int(total),
        "nonzero": nonzero,
        "dead": dead,
        "util_pct": util_pct,
        "entropy_nats": entropy,
        "perplexity": perplexity,
    }
    for k in topk_list:
        out[f"top{k}_mass"] = top_mass[k]
    return out

def print_stats_table(model_name: str, counts_dict: Dict[str, np.ndarray]) -> None:
    print(f"\n{model_name} | token distribution stats")
    top_headers = " | ".join([f"{('top'+str(k)+'%'):>8}" for k in TOPK_LIST])
    header = f"{'Level':<16} | {'util%':>6} | {'nonzero':>6} | {'H(nats)':>8} | {'PPL':>9} | {top_headers}"
    print(header)
    print("-" * len(header))

    for label in COUNT_LABELS:
        s = token_distribution_stats(counts_dict[label])
        tops = " | ".join([f"{100.0*s[f'top{k}_mass']:8.2f}" for k in TOPK_LIST])
        print(f"{label:<16} | {s['util_pct']:6.2f} | {s['nonzero']:6d} | {s['entropy_nats']:8.3f} | {s['perplexity']:9.1f} | {tops}")

print_stats_table("VQGAN", vq_counts)
print_stats_table("LlamaGen", ll_counts)


Interpretation: the edit operation is behaving exactly as designed. We are editing almost exclusively the intended patch, with tiny discrete deviations explainable by bbox rounding / clipping near image borders.